# 作业2：基于多模态因子双线性池化（MFB）的视觉问答 — PyTorch实现

**目标**：将PaddlePaddle版本的MFBVQA视觉问答模型转换为PyTorch实现，在CPU上运行。

**要求**：
- 补全标有 `# TODO` 的代码块
- 所有代码基于PyTorch，不使用PaddlePaddle
- 默认在CPU上运行

**模型概述**：MFBVQA模型使用MFB（多模态因子双线性池化）进行跨模态融合，通过多头注意力将问题和图像区域对齐，最终预测答案。

## 环境准备

In [ ]:
!pip install torch numpy pillow matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
import os
import random
import re
from os.path import join as pjoin
from argparse import Namespace
from collections import defaultdict, Counter

device = torch.device('cpu')
print(f'使用设备: {device}')
print(f'PyTorch版本: {torch.__version__}')

## 读取数据

以下数据处理代码已提供，无需修改。

In [ ]:
!tar -zxf ./data/data245864/vqa_processed.tar.gz -C ./data
!mv ./data/vqa_processed ./data/vqa

In [ ]:
import base64
import csv
import sys
from PIL import Image

csv.field_size_limit(sys.maxsize)

def tokenize_mcb(s):
    t_str = s.lower()
    for i in [r'\?',r'\!',r'\'',r'\"',r'\$',r'\:',r'\@',r'\(',r'\)',r'\,',r'\.',r'\;']:
        t_str = re.sub(i, '', t_str)
    for i in [r'\-',r'\/']:
        t_str = re.sub(i, ' ', t_str)
    q_list = re.sub(r'\?','',t_str.lower()).split(' ')
    q_list = list(filter(lambda x: len(x) > 0, q_list))
    return q_list

def tokenize_questions(questions):
    for item in questions:
        item['question_tokens'] = tokenize_mcb(item['question'])
    return questions

def annotations_in_top_answers(annotations, questions, ans_vocab):
    new_anno, new_ques = [], []
    assert len(annotations) == len(questions)
    for anno, ques in zip(annotations, questions):
        if anno['multiple_choice_answer'] in ans_vocab:
            new_anno.append(anno)
            new_ques.append(ques)
    return new_anno, new_ques

def encode_questions(questions, vocab):
    for item in questions:
        item['question_idx'] = [vocab.get(w, vocab['<unk>']) for w in item['question_tokens']]
    return questions

def encode_answers(annotations, vocab):
    for item in annotations:
        item['answer_list'] = []
        answers = [a['answer'] for a in item['answers']]
        for ans in answers:
            if ans in vocab:
                item['answer_list'].append(vocab[ans])
    return annotations

In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt

data_dir = './data/vqa/vqa2/'
dir_processed = pjoin(data_dir, 'processed')
rcnn_dir = './data/vqa/coco/image_box_features/'
image_dir = './data/vqa/coco/raw/val2014/'

vocab = json.load(open(pjoin(dir_processed, 'vocab.json'), 'r'))
dataset = json.load(open(pjoin(dir_processed, 'val_data.json'), 'r'))

idx2ans = {i: a for a, i in vocab['ans_vocab'].items()}
idx2ques = {i: q for q, i in vocab['ques_vocab'].items()}

idx = 10000
question = dataset['questions'][idx]
q_text = ' '.join([idx2ques[token] for token in question['question_idx']])
annotation = dataset['annotations'][idx]
a_text = '/'.join([idx2ans[token] for token in annotation['answer_list']])
print(f'问题: {q_text}')
print(f'回答: {a_text}')

### 任务1：定义数据集类和collate_fn（10分）

将PaddlePaddle的数据集类转换为PyTorch版本。

**提示**：
- `paddle.io.Dataset` → `torch.utils.data.Dataset`
- `paddle.to_tensor(x, dtype='int64')` → `torch.tensor(x, dtype=torch.long)`
- `paddle.zeros()` → `torch.zeros()`

In [ ]:
def collate_fn(batch):
    """对一个批次的数据进行预处理，将变长问题补齐到同一长度"""
    max_question_length = max([len(item['question']) for item in batch])
    batch_size = len(batch)

    # TODO: 创建批次张量
    # 提示：
    #   imgs: torch.zeros((batch_size, 36, 2048))  — 图像区域特征
    #   ques: torch.zeros((batch_size, max_question_length), dtype=torch.long)  — 问题索引
    #   ans:  torch.zeros((batch_size, 1000))  — 回答分布
    #   lens: torch.zeros(batch_size, dtype=torch.long)  — 问题长度
    imgs = None  # 请替换
    ques = None  # 请替换
    ans = None   # 请替换
    lens = None  # 请替换

    for i, item in enumerate(batch):
        # TODO: 填充每个样本的数据
        # 提示：
        #   imgs[i] = torch.from_numpy(item['image_feat'])
        #   ques[i, :item['question'].shape[0]] = item['question']
        #   for answer in item['answers']: ans[i, answer] += 1
        #   lens[i] = item['length']
        pass  # 请补全

    return (imgs, ques, ans, lens)


class VQA2Dataset(Dataset):

    def __init__(self,
            data_dir='./data/vqa/vqa2/',
            rcnn_dir='./data/vqa/coco/image_box_features/',
            split='train',
            samplingans=True):
        super(VQA2Dataset, self).__init__()
        self.rcnn_dir = rcnn_dir
        self.samplingans = samplingans
        self.split = split
        dir_processed = pjoin(data_dir, 'processed')
        if split == 'train':
            self.dataset = json.load(open(pjoin(dir_processed, 'train_data.json'), 'r'))
        elif split == 'val':
            self.dataset = json.load(open(pjoin(dir_processed, 'val_data.json'), 'r'))
        self.dataset_size = len(self.dataset['questions'])

    def __getitem__(self, index):
        item = {}
        item['index'] = index

        question = self.dataset['questions'][index]

        # TODO: 将问题和长度转为PyTorch张量
        # 提示：
        #   item['question'] = torch.tensor(question['question_idx'], dtype=torch.long)
        #   item['length'] = torch.tensor([len(question['question_idx'])], dtype=torch.long)
        item['question'] = None  # 请替换
        item['length'] = None    # 请替换

        item['image_feat'] = np.load(pjoin(self.rcnn_dir, '{}.jpg.npy'.format(question['image_id'])))

        annotation = self.dataset['annotations'][index]
        if 'train' in self.split and self.samplingans:
            item['answers'] = [random.choice(annotation['answer_list'])]
        else:
            item['answers'] = annotation['answer_list']
        return item

    def __len__(self):
        return self.dataset_size

In [ ]:
def mktrainval(data_dir, image_feat_dir, batch_size, workers=0):
    train_set = VQA2Dataset(data_dir, image_feat_dir, split='train', samplingans=True)
    valid_set = VQA2Dataset(data_dir, image_feat_dir, split='val', samplingans=False)

    train_loader = DataLoader(
                        train_set, batch_size=batch_size,
                        shuffle=True, num_workers=workers,
                        collate_fn=collate_fn)
    valid_loader = DataLoader(
                        valid_set, batch_size=batch_size,
                        shuffle=False, num_workers=workers,
                        drop_last=False, collate_fn=collate_fn)

    return train_loader, valid_loader

## 定义模型

### 任务2：实现MFB融合（10分）

MFB（Multimodal Factorized Bilinear）融合操作：将两个不同模态的表示通过低秩双线性池化进行融合。

**提示**：
- `nn.Layer` → `nn.Module`
- `.dim()` 方法在PyTorch中用法相同
- `.reshape()` 用法相同
- `.sum(3).squeeze(1)` 用法相同

In [ ]:
class MFBFusion(nn.Module):
    def __init__(self, input_dim1, input_dim2, hidden_dim, R):
        '''
        参数：
            input_dim1: 第一个待融合表示的维度
            input_dim2: 第二个待融合表示的维度
            hidden_dim: 融合后的表示的维度
            R: MFB所使用的低秩矩阵的数量
        '''
        super(MFBFusion, self).__init__()
        self.input_dim1 = input_dim1
        self.input_dim2 = input_dim2
        self.hidden_dim = hidden_dim
        self.R = R

        # TODO: 定义两个线性层
        # 提示：
        #   self.linear1 = nn.Linear(input_dim1, hidden_dim * R)
        #   self.linear2 = nn.Linear(input_dim2, hidden_dim * R)
        self.linear1 = None  # 请替换
        self.linear2 = None  # 请替换

    def forward(self, inputs1, inputs2):
        '''
        参数：
            inputs1: (batch_size, input_dim1) 或 (batch_size, num_region, input_dim1)
            inputs2: (batch_size, input_dim2) 或 (batch_size, num_region, input_dim2)
        返回：
            z: (batch_size, hidden_dim) 或 (batch_size, num_region, hidden_dim)
        '''
        num_region = 1
        if inputs1.dim() == 3:
            num_region = inputs1.shape[1]

        # TODO: 实现MFB融合
        # 步骤：
        #   1. h1 = self.linear1(inputs1)
        #   2. h2 = self.linear2(inputs2)
        #   3. 逐元素相乘: z = h1 * h2
        #   4. reshape为 (batch, num_region, hidden_dim, R)
        #   5. 在R维度上求和: z.sum(3).squeeze(1)

        z = None  # 请替换
        return z

### 任务3：实现多头注意力跨模态对齐（20分）

实现基于MFB的多头交叉注意力模块。这是本作业的核心部分。

**提示**：
- `nn.LayerList` → `nn.ModuleList`（很重要！否则子模块不会被正确注册）
- `nn.Softmax(axis=1)` → `nn.Softmax(dim=1)`
- `paddle.repeat_interleave(x, n, axis)` → `torch.repeat_interleave(x, n, dim=axis)`
- `paddle.transpose(alphas, (0,2,1))` → `alphas.permute(0, 2, 1)`
- `paddle.bmm()` → `torch.bmm()`
- `paddle.split(output, output.shape[1], axis=1)` 按份数切分 → `torch.split(output, 1, dim=1)` 按每份大小切分
- `paddle.concat(list, axis)` → `torch.cat(list, dim=axis)`

In [ ]:
class MultiHeadATTN(nn.Module):
    def __init__(self, query_dim, kv_dim, mfb_input_dim, mfb_hidden_dim, num_head, att_dim):
        super(MultiHeadATTN, self).__init__()
        assert att_dim % num_head == 0
        self.num_head = num_head
        self.att_dim = att_dim

        self.attn_w_1_q = nn.Sequential(
                            nn.Dropout(0.5),
                            nn.Linear(query_dim, mfb_input_dim),
                            nn.ReLU()
                          )
        self.attn_w_1_k = nn.Sequential(
                            nn.Dropout(0.5),
                            nn.Linear(kv_dim, mfb_input_dim),
                            nn.ReLU()
                          )
        self.attn_score_fusion = MFBFusion(mfb_input_dim, mfb_input_dim, mfb_hidden_dim, 1)
        self.attn_score_mapping = nn.Sequential(
                            nn.Dropout(0.5),
                            nn.Linear(mfb_hidden_dim, num_head)
                          )

        # TODO: 定义softmax层
        # 提示：nn.Softmax(dim=1)  — Paddle中是 axis=1
        self.softmax = None  # 请替换

        # TODO: 定义对齐表示计算层列表
        # 提示：使用 nn.ModuleList（不是nn.LayerList！）
        #   nn.ModuleList([nn.Sequential(
        #       nn.Dropout(0.5),
        #       nn.Linear(kv_dim, int(att_dim / num_head)),
        #       nn.Tanh()
        #   ) for _ in range(num_head)])
        self.align_q = None  # 请替换

    def forward(self, query, key_value):
        """
        参数：
          query: (batch_size, q_dim)
          key_value: (batch_size, num_region, kv_dim)
        返回：
          align_feat: (batch_size, att_dim) 对齐后的特征
          alpha: 注意力权重列表
        """
        num_region = key_value.shape[1]

        # TODO (步骤1): 将query扩展到每个区域
        # 提示：
        #   q = torch.repeat_interleave(self.attn_w_1_q(query).unsqueeze(1), num_region, dim=1)
        #   注意：Paddle中 axis=1 → PyTorch中 dim=1
        q = None  # 请替换

        k = self.attn_w_1_k(key_value)

        # TODO (步骤2): 计算注意力得分并归一化
        # 提示：
        #   alphas = self.attn_score_fusion(q, k)
        #   alphas = self.attn_score_mapping(alphas)
        #   alphas = self.softmax(alphas)
        alphas = None  # 请替换

        # TODO (步骤3): 计算加权输出
        # 提示：
        #   output = torch.bmm(alphas.permute(0, 2, 1), key_value)
        #   注意：Paddle用 paddle.transpose(alphas, (0,2,1)) → PyTorch用 alphas.permute(0, 2, 1)
        output = None  # 请替换

        # TODO (步骤4): 分割多头并拼接
        # 提示：
        #   Paddle: paddle.split(output, output.shape[1], axis=1) 第二个参数是份数
        #   PyTorch: torch.split(output, 1, dim=1) 第二个参数是每份大小
        #   list_v = [e.squeeze(1) for e in torch.split(output, 1, dim=1)]
        #   alpha = torch.split(alphas, 1, dim=2)
        #   align_feat = torch.cat([self.align_q[head_id](x_v) for head_id, x_v in enumerate(list_v)], dim=1)
        align_feat = None  # 请替换
        alpha = None       # 请替换

        return align_feat, alpha

### 任务4：实现MFBVQA模型（20分）

实现完整的MFBVQA模型，包含GRU文本编码器、多头注意力对齐和MFB融合。

**提示（GRU变长序列处理 — 最关键的区别）**：
- Paddle的GRU直接支持 `sequence_length` 参数
- PyTorch需要使用 `pack_padded_sequence` + `pad_packed_sequence` 处理变长序列：
  1. 先按长度降序排序
  2. `packed = nn.utils.rnn.pack_padded_sequence(sorted_x, sorted_lengths, batch_first=True)`
  3. `_, hidden = self.text_encoder(packed)`
  4. 恢复原始顺序
- `nn.initializer.Uniform(low=-0.1, high=0.1)` → `nn.init.uniform_(self.embed.weight, -0.1, 0.1)`
- `nn.functional.normalize(x)` → `F.normalize(x, dim=1)`（PyTorch需显式指定dim）

In [ ]:
class MFBVQAModel(nn.Module):
    def __init__(self, vocab_words, question_dim, image_dim,
                       attn_mfb_input_dim, attn_mfb_hidden_dim,
                       attn_num_head, attn_output_dim,
                       fusion_q_feature_dim, fusion_mfb_hidden_dim,
                       num_classes):
        super(MFBVQAModel, self).__init__()

        # TODO: 初始化嵌入层，权重用均匀分布 [-0.1, 0.1] 初始化
        # 提示：
        #   self.embed = nn.Embedding(len(vocab_words), 300)
        #   nn.init.uniform_(self.embed.weight, -0.1, 0.1)
        self.embed = None  # 请替换

        # TODO: 初始化GRU文本编码器
        # 提示：nn.GRU(300, question_dim, num_layers=2, batch_first=True)
        self.text_encoder = None  # 请替换

        self.attn = MultiHeadATTN(question_dim, image_dim,
                                  attn_mfb_input_dim, attn_mfb_hidden_dim,
                                  attn_num_head, attn_output_dim)

        self.q_feature_linear = nn.Sequential(
                                    nn.Dropout(0.5),
                                    nn.Linear(question_dim, fusion_q_feature_dim),
                                    nn.ReLU()
                                )

        self.fusion = MFBFusion(attn_output_dim, fusion_q_feature_dim, fusion_mfb_hidden_dim, 2)

        self.classifier_linear = nn.Sequential(
                                    nn.Dropout(0.5),
                                    nn.Linear(fusion_mfb_hidden_dim, num_classes)
                                )
        self.question_dim = question_dim

    def forward(self, imgs, quests, lengths):
        v_feature = imgs.reshape((-1, 36, 2048))
        x = self.embed(quests)

        # TODO: 使用GRU编码问题（处理变长序列）
        # 这是Paddle→PyTorch最关键的区别！
        # Paddle: _, hidden = self.text_encoder(x, None, lengths)
        # PyTorch步骤：
        #   1. 按长度降序排序:
        #      lengths_cpu = lengths.cpu()
        #      sorted_lengths, sorted_idx = torch.sort(lengths_cpu, descending=True)
        #      sorted_x = x[sorted_idx]
        #      sorted_lengths = sorted_lengths.clamp(min=1)
        #   2. 打包:
        #      packed = nn.utils.rnn.pack_padded_sequence(sorted_x, sorted_lengths.tolist(), batch_first=True)
        #   3. 编码:
        #      _, hidden = self.text_encoder(packed)
        #   4. 恢复原始顺序:
        #      _, unsorted_idx = torch.sort(sorted_idx)
        #      hidden = hidden[:, unsorted_idx, :]

        hidden = None  # 请替换（得到shape为 (num_layers, batch, hidden_size) 的hidden）

        # TODO: 取最后一层隐藏状态并L2归一化
        # 提示：q_feature = F.normalize(hidden[-1], dim=1)
        # 注意：Paddle中 nn.functional.normalize(x) 不需要指定dim，PyTorch需要
        q_feature = None  # 请替换

        # 利用注意力获得问题的对齐表示
        align_q_feature, _ = self.attn(q_feature, v_feature)
        # 对原始文本表示进行变换
        original_q_feature = self.q_feature_linear(q_feature)
        # 融合对齐前后的问题的表示
        x = self.fusion(align_q_feature, original_q_feature)
        # 分类
        x = self.classifier_linear(x)
        return x

### 任务5：定义损失函数（5分）

**提示**：
- `nn.Layer` → `nn.Module`
- `nn.functional.log_softmax(input)` → `F.log_softmax(input, dim=-1)`（PyTorch必须指定dim）

In [ ]:
class KLLoss(nn.Module):
    def __init__(self):
        super(KLLoss, self).__init__()
        self.loss = nn.KLDivLoss(reduction='batchmean')

    def forward(self, input, target):
        # TODO: 计算KL散度损失
        # 提示：
        #   Paddle: self.loss(nn.functional.log_softmax(input), target)
        #   PyTorch: self.loss(F.log_softmax(input, dim=-1), target)
        #   注意：PyTorch的log_softmax必须显式指定dim参数

        return None  # 请替换

### 任务6：选择优化方法（5分）

**提示**：
- `paddle.optimizer.lr.ExponentialDecay` → `torch.optim.lr_scheduler.ExponentialLR`
- `paddle.optimizer.Adam` → `torch.optim.Adam`
- PyTorch中优化器和学习率调度器是分开创建的

In [ ]:
def get_optimizer(model, config):
    """创建优化器和学习率调度器"""
    # TODO: 创建Adam优化器和ExponentialLR学习率调度器
    # 提示：
    #   optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    #   scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.5 ** (1 / 50000))

    optimizer = None   # 请替换
    scheduler = None   # 请替换

    return optimizer, scheduler

### 任务7：实现评估函数（10分）

**提示**：
- `paddle.arange()` → `torch.arange()`
- `output.argmax(axis=1)` → `output.argmax(dim=1)`
- 使用 `torch.no_grad()` 上下文管理器减少内存占用
- `hit_ct.item()` 获取标量值

In [ ]:
def evaluate(data_loader, model):
    model.eval()
    accs = []

    # TODO: 实现评估逻辑
    # 步骤：
    #   1. 在 torch.no_grad() 下遍历 data_loader
    #   2. 将数据移到device: imgs.to(device), questions.to(device), etc.
    #   3. 前馈: output = model(imgs, questions, lengths)
    #   4. 计算准确率:
    #      hit_cts = answers[torch.arange(output.shape[0]), output.argmax(dim=1)]
    #      for hit_ct in hit_cts:
    #          accs.append(min(1, hit_ct.item() / 3.0))

    pass  # 请补全

    model.train()
    return float(sum(accs)) / len(accs) if len(accs) > 0 else 0.0

### 任务8：完成训练循环（20分）

**提示**：
- `optimizer.clear_grad()` → `optimizer.zero_grad()`
- 需要额外调用 `scheduler.step()` 来更新学习率
- `nn.utils.clip_grad_norm_()` 用法相同
- `paddle.save()` → `torch.save()`
- 使用 `.to(device)` 将数据送到指定设备

In [ ]:
config = Namespace(
    question_dim=2400,
    image_dim=2048,
    attn_mfb_input_dim=310,
    attn_mfb_hidden_dim=510,
    attn_num_head=2,
    attn_output_dim=620,
    fusion_q_feature_dim=310,
    fusion_mfb_hidden_dim=510,
    num_ans=1000,
    batch_size=128,
    learning_rate=0.0001,
    margin=0.2,
    num_epochs=45,
    grad_clip=0.25,
    evaluate_step=360,
    checkpoint=None,
    best_checkpoint='./model/mfb/best_vqa2.ckpt',
    last_checkpoint='./model/mfb/last_vqa2.ckpt'
)

data_dir = './data/vqa/vqa2/'
dir_processed = pjoin(data_dir, 'processed')

train_loader, valid_loader = mktrainval(data_dir,
               './data/vqa/coco/image_box_features/',
               config.batch_size,
               workers=0)

vocab = json.load(open(pjoin(dir_processed, 'vocab.json'), 'r'))

model = MFBVQAModel(vocab['ques_vocab'],
                    config.question_dim,
                    config.image_dim,
                    config.attn_mfb_input_dim,
                    config.attn_mfb_hidden_dim,
                    config.attn_num_head,
                    config.attn_output_dim,
                    config.fusion_q_feature_dim,
                    config.fusion_mfb_hidden_dim,
                    config.num_ans)
model = model.to(device)

optimizer, scheduler = get_optimizer(model, config)
model.train()
loss_fn = KLLoss()

os.makedirs(os.path.dirname(config.best_checkpoint), exist_ok=True)

best_res = 0
print("开始训练")
fw = open('log.txt', 'w')

for epoch in range(config.num_epochs):
    for i, (imgs, questions, answers, lengths) in enumerate(train_loader):
        # TODO: 完成训练步骤
        # 步骤：
        #   1. 将数据移到device:
        #      imgs = imgs.to(device)
        #      questions = questions.to(device)
        #      answers = answers.to(device)
        #      lengths = lengths.to(device)
        #   2. optimizer.zero_grad()  — 清零梯度（Paddle中是 optimizer.clear_grad()）
        #   3. 前馈: output = model(imgs, questions, lengths)
        #   4. 计算损失: loss = loss_fn(output, answers)
        #   5. loss.backward()  — 反向传播
        #   6. 梯度裁剪: nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        #   7. optimizer.step()  — 更新参数
        #   8. scheduler.step()  — 更新学习率（Paddle中集成在optimizer里，PyTorch需要单独调用）

        pass  # 请补全

        state = {
                'epoch': epoch,
                'step': i,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict()
                }

        if (i + 1) % config.evaluate_step == 0:
            acc = evaluate(valid_loader, model)
            if best_res < acc:
                best_res = acc
                torch.save(state, config.best_checkpoint)
            torch.save(state, config.last_checkpoint)
            log_msg = 'epoch: %d, step: %d, loss: %.2f, ACC: %.3f' % (
                epoch, i + 1, loss.item(), acc)
            print(log_msg)
            fw.write(log_msg + '\n')
fw.close()

## Paddle → PyTorch 对照速查表

| Paddle | PyTorch | 说明 |
|--------|---------|------|
| `nn.Layer` | `nn.Module` | 模型基类 |
| `nn.LayerList` | `nn.ModuleList` | 模块列表（确保子模块被正确注册） |
| `paddle.io.Dataset` | `torch.utils.data.Dataset` | 数据集基类 |
| `paddle.to_tensor(x, dtype='int64')` | `torch.tensor(x, dtype=torch.long)` | 创建张量 |
| `paddle.zeros(shape)` | `torch.zeros(shape)` | 零张量 |
| `paddle.repeat_interleave(x, n, axis)` | `torch.repeat_interleave(x, n, dim=axis)` | `axis`→`dim` |
| `paddle.split(x, 份数, axis)` | `torch.split(x, 每份大小, dim)` | 参数含义不同！ |
| `paddle.transpose(x, perm)` | `x.permute(*perm)` | 维度变换 |
| `paddle.bmm()` | `torch.bmm()` | 批矩阵乘 |
| `paddle.concat(list, axis)` | `torch.cat(list, dim=axis)` | 拼接 |
| `nn.Softmax(axis=1)` | `nn.Softmax(dim=1)` | `axis`→`dim` |
| `F.normalize(x)` | `F.normalize(x, dim=1)` | 需指定dim |
| `F.log_softmax(x)` | `F.log_softmax(x, dim=-1)` | 需指定dim |
| GRU + `sequence_length` | GRU + `pack_padded_sequence` | 变长序列处理方式不同 |
| `nn.initializer.Uniform` | `nn.init.uniform_` | 均匀初始化 |
| `optimizer.clear_grad()` | `optimizer.zero_grad()` | 清除梯度 |
| lr集成在optimizer中 | `scheduler.step()` 单独调用 | 学习率调度 |
| `paddle.save()` | `torch.save()` | 保存模型 |
| 无 | `torch.no_grad()` | 评估时关闭梯度 |